
# Analisis Data Indeks Standar Pencemar Udara (ISPU) DKI Jakarta 2023

Dataset ini bersumber dari portal data resmi Indonesia (data.go.id). 
Menurut deskripsi dataset, **Indeks Standar Pencemar Udara (ISPU)** diukur melalui Stasiun Pemantau Kualitas Udara (SPKU) di Provinsi DKI Jakarta. 
Data ini merekam konsentrasi berbagai polutan (PM10, PM2.5, sulfur dioksida, karbon monoksida, ozon, nitrogen dioksida) pada beberapa stasiun pemantauan, 
tanggal pengukuran, nilai indeks polutan tertinggi (`max`), parameter pencemar kritis (`parameter_pencemar_kritis`), serta kategori kualitas udara. 
Dataset ini diterbitkan tanpa preprocessing, sehingga cocok untuk eksplorasi data dan latihan tahap‐tahap preprocessing (cleaning, transformation, reduction).

Tujuan dari notebook ini:
1. Memahami struktur dan karakteristik data (Data Understanding).
2. Melakukan analisis data eksploratif (EDA) dengan statistik deskriptif dan visualisasi.
3. Melakukan preprocessing dalam tiga tahap: **Data Cleaning**, **Data Transformation**, dan **Data Reduction**.
4. Menyimpulkan temuan dari analisis.


## 1. Muat Dataset dan Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
import io, os

%matplotlib inline

# ── Muat Dataset ────────────────────────────────────────────────────────────
# Di Google Colab: klik "Choose Files" lalu pilih
#   "Dataset Pencemaran Udara DKI JAKARTA.csv"
# Di Jupyter lokal: pastikan file CSV berada di folder yang sama dengan notebook

_CSV_NAME = "Dataset Pencemaran Udara DKI JAKARTA.csv"

try:
    from google.colab import files as _colab_files
    print("Jalankan di Google Colab — silakan upload file CSV:")
    _uploaded = _colab_files.upload()   # tampilkan widget upload
    _fname = list(_uploaded.keys())[0]
    ispu_df = pd.read_csv(
        io.BytesIO(_uploaded[_fname]),
        encoding="utf-8-sig",
        na_values=["-", "---"],
    )
except ImportError:
    # Jupyter lokal — baca dari file di folder yang sama
    ispu_df = pd.read_csv(
        _CSV_NAME,
        encoding="utf-8-sig",
        na_values=["-", "---"],
    )

print(f"Bentuk data: {ispu_df.shape}")
print(ispu_df.head())



## 2. Pemahaman Data (Data Understanding)

Kita akan meninjau struktur data dengan `info()` dan menghitung statistik deskriptif dasar. Data memiliki kolom-kolom berikut:

- `periode_data` : kode periode (YYYYMM) yang menunjukkan bulan dan tahun pengukuran.
- `tanggal` : tanggal pengukuran (format YYYY-MM-DD).
- `stasiun` : nama Stasiun Pemantau Kualitas Udara (SPKU) di DKI Jakarta.
- `pm_sepuluh` : konsentrasi partikel debu kasar (PM10) dalam μg/m³.
- `pm_duakomalima` : konsentrasi partikel debu halus (PM2.5) dalam μg/m³.
- `sulfur_dioksida` : konsentrasi SO₂ dalam μg/m³.
- `karbon_monoksida` : konsentrasi CO dalam mg/m³.
- `ozon` : konsentrasi O₃ dalam μg/m³.
- `nitrogen_dioksida` : konsentrasi NO₂ dalam μg/m³.
- `max` : nilai maksimum dari indeks polutan untuk masing-masing parameter.
- `parameter_pencemar_kritis` : parameter polutan yang memiliki indeks tertinggi pada hari tersebut.
- `kategori` : kategori mutu udara berdasarkan nilai indeks (BAIK, SEDANG, TIDAK SEHAT, dll.).

Berikut hasil pemeriksaan struktur dan statistik deskriptif:


In [ ]:

# Struktur data
ispu_df.info()

# Statistik deskriptif untuk kolom numerik
numeric_cols = ['pm_sepuluh', 'pm_duakomalima', 'sulfur_dioksida', 'karbon_monoksida', 'ozon', 'nitrogen_dioksida', 'max']
print(ispu_df[numeric_cols].describe())

# Cek nilai hilang per kolom
print("\nJumlah nilai hilang per kolom:")
print(ispu_df.isna().sum())



## 3. Analisis Data Eksploratif (EDA)

Pada bagian ini kita akan membuat visualisasi sederhana untuk memahami distribusi nilai polutan dan pola kualitas udara.
Beberapa plot yang disajikan:

1. Histogram distribusi nilai **`max`** (indeks polutan tertinggi).
2. Bar chart jumlah observasi untuk setiap kategori mutu udara (`kategori`).
3. Bar chart rata-rata PM2.5 (`pm_duakomalima`) per stasiun.


In [ ]:

# 1. Histogram nilai max
plt.figure(figsize=(8,5))
sns.histplot(ispu_df['max'], bins=30, kde=True)
plt.title('Distribusi Nilai Indeks Polutan Tertinggi (max)')
plt.xlabel('Nilai Index')
plt.ylabel('Frekuensi')
plt.tight_layout()
plt.show()

# 2. Jumlah observasi per kategori kualitas udara
plt.figure(figsize=(6,4))
kat_counts = ispu_df['kategori'].value_counts()
sns.barplot(x=kat_counts.index, y=kat_counts.values)
plt.title('Jumlah Observasi per Kategori Kualitas Udara')
plt.xlabel('Kategori')
plt.ylabel('Jumlah Observasi')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 3. Rata-rata PM2.5 per stasiun
plt.figure(figsize=(10,6))
station_avg_pm25 = ispu_df.groupby('stasiun')['pm_duakomalima'].mean().sort_values(ascending=False).head(10)
sns.barplot(y=station_avg_pm25.index, x=station_avg_pm25.values)
plt.title('10 Stasiun dengan Rata-rata PM2.5 Tertinggi')
plt.xlabel('Rata-rata PM2.5 (µg/m³)')
plt.ylabel('Stasiun')
plt.tight_layout()
plt.show()



## 4. Preprocessing Tahap 1: Data Cleaning

Langkah-langkah pembersihan yang dilakukan:

1. **Mengubah tipe data**: kolom `tanggal` diubah ke format `datetime` agar memudahkan ekstraksi tahun/bulan.
2. **Menghilangkan duplikat**: jika ada baris data yang duplikat, kita hapus.
3. **Menangani nilai hilang**: dataset ini relatif lengkap, tetapi kita akan melakukan imputasi menggunakan median untuk kolom numerik jika ada nilai hilang.
4. **Membersihkan kolom teks**: membersihkan spasi berlebih pada nama stasiun.

Berikut implementasinya:


In [ ]:

# Membuat salinan data untuk diproses
clean_df = ispu_df.copy()

# 1. Ubah tanggal ke datetime
clean_df['tanggal'] = pd.to_datetime(clean_df['tanggal'])

# 2. Hilangkan baris duplikat
clean_df = clean_df.drop_duplicates()

# Hapus baris tanpa data (kategori "TIDAK ADA DATA")
clean_df = clean_df[clean_df["kategori"] != "TIDAK ADA DATA"].reset_index(drop=True)

# 3. Imputasi nilai hilang (jika ada) dengan median untuk numerik
num_cols = ['pm_sepuluh', 'pm_duakomalima', 'sulfur_dioksida', 'karbon_monoksida', 'ozon', 'nitrogen_dioksida', 'max']
num_imputer = SimpleImputer(strategy='median')
clean_df[num_cols] = num_imputer.fit_transform(clean_df[num_cols])

# 4. Membersihkan spasi berlebih pada kolom stasiun
clean_df['stasiun'] = clean_df['stasiun'].str.strip()

# Cek ulang dimensi data setelah cleaning
print(f"Dimensi data setelah cleaning: {clean_df.shape}")
print("Jumlah nilai hilang setelah cleaning:")
print(clean_df.isna().sum())



## 5. Preprocessing Tahap 2: Data Transformation

Transformasi bertujuan untuk mengubah data agar siap digunakan oleh model machine learning atau analisis lebih lanjut.
Langkah-langkah yang dilakukan:

1. **Ekstraksi fitur waktu**: membuat kolom baru `tahun` dan `bulan` dari kolom `tanggal`.
2. **Transformasi log**: mengaplikasikan transformasi log (menggunakan `np.log1p`) pada kolom numerik untuk mengurangi skewness.
3. **Standarisasi**: menstandarisasi kolom numerik agar memiliki mean 0 dan standar deviasi 1 menggunakan `StandardScaler`.
4. **Pengkodean satu-satu (One-Hot Encoding)**: mengubah kolom kategori (`kategori` dan `parameter_pencemar_kritis`) menjadi fitur dummy.

Berikut implementasinya:


In [ ]:

# Salin data cleaned untuk transformasi
trans_df = clean_df.copy()

# 1. Ekstraksi tahun dan bulan
trans_df['tahun'] = trans_df['tanggal'].dt.year
trans_df['bulan'] = trans_df['tanggal'].dt.month

# 2. Transformasi log pada kolom numerik (hindari 0 menggunakan log1p)
for col in num_cols:
    trans_df[f'log_{col}'] = np.log1p(trans_df[col])

# 3. Standarisasi kolom numerik (log-transformed)
log_num_cols = [f'log_{col}' for col in num_cols]
scaler = StandardScaler()
trans_df[log_num_cols] = scaler.fit_transform(trans_df[log_num_cols])

# 4. One-Hot Encoding untuk fitur kategorikal
categorical_cols = ['kategori', 'parameter_pencemar_kritis', 'stasiun']
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded_cats = encoder.fit_transform(trans_df[categorical_cols])

# Membuat DataFrame hasil encoding
encoded_cols = encoder.get_feature_names_out(categorical_cols)
encoded_df = pd.DataFrame(encoded_cats, columns=encoded_cols, index=trans_df.index)

# Gabungkan data numerik terstandardisasi dan fitur waktu dengan encoded categories
transformed_df = pd.concat([trans_df[['tahun','bulan']], trans_df[log_num_cols], encoded_df], axis=1)

print(f"Dimensi data setelah transformasi: {transformed_df.shape}")



## 6. Preprocessing Tahap 3: Data Reduction

Reduksi dimensi bertujuan menurunkan jumlah fitur dengan tetap mempertahankan informasi sebanyak mungkin. Dua teknik yang diterapkan:

1. **Variance Threshold**: menghapus fitur dengan variasi rendah (varian < 0.01), karena fitur tersebut cenderung tidak membawa informasi.
2. **Principal Component Analysis (PCA)**: mereduksi dimensi menjadi beberapa komponen utama dan melihat proporsi varians yang dijelaskan.

Berikut implementasinya:


In [ ]:

# 1. Variance Threshold
selector = VarianceThreshold(threshold=0.01)
reduced_var_df = selector.fit_transform(transformed_df)

print(f"Dimensi setelah Variance Threshold: {reduced_var_df.shape}")

# Mendapatkan nama fitur yang dipertahankan
retained_features = transformed_df.columns[selector.get_support(indices=True)]

# 2. PCA
pca = PCA(n_components=2)
pca_components = pca.fit_transform(reduced_var_df)
print("Proporsi varians yang dijelaskan oleh 2 komponen utama:", pca.explained_variance_ratio_)

# Scatter plot dari dua komponen PCA
plt.figure(figsize=(6,5))
plt.scatter(pca_components[:,0], pca_components[:,1], alpha=0.5)
plt.title('Visualisasi 2 Komponen PCA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.tight_layout()
plt.show()



## 7. Kesimpulan

Dalam notebook ini kita telah melakukan eksplorasi dan preprocessing pada dataset **Indeks Standar Pencemar Udara (ISPU)** di Provinsi DKI Jakarta tahun 2023.
Beberapa poin penting:

- Dataset terdiri dari 1825 baris dan 12 kolom, mencakup tanggal, lokasi stasiun, nilai beberapa polutan, indeks polutan tertinggi, serta kategori kualitas udara.
- EDA menunjukkan distribusi nilai indeks polutan tertinggi dan variasi kategori mutu udara. Rata-rata PM2.5 tertinggi ditemukan di beberapa stasiun tertentu.
- Pada tahap pembersihan, kita konversi tipe tanggal, menghapus duplikat, melakukan imputasi median untuk nilai hilang, dan merapikan nama stasiun.
- Transformasi dilakukan dengan ekstraksi fitur waktu, transformasi log agar data lebih normal, standarisasi, dan encoding variabel kategorikal.
- Reduksi dimensi menggunakan Variance Threshold dan PCA berhasil menurunkan jumlah fitur; dua komponen utama menjelaskan sebagian varians. Visualisasi scatter plot PCA memperlihatkan sebaran data dalam ruang dua dimensi.

Dataset ini kini siap untuk analisis lebih lanjut atau pembuatan model prediksi kualitas udara, misalnya klasifikasi kategori kualitas udara berdasarkan polutan, atau analisis pola musiman.
